# 05 - Live Trading Bridge

Take target weights through the **execution layer**: pre-trade risk checks, weight->order translation, the order-lifecycle state machine, state persistence, and an Alpaca **paper** connection (auto-skipped offline). One full rebalance cycle.

In [1]:
import os
import sys
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
# hmmlearn reports non-convergence via logging (not warnings), so it
# bypasses the filter above; quiet it for clean notebook output.
import logging

logging.getLogger("hmmlearn").setLevel(logging.ERROR)
import matplotlib

matplotlib.use("Agg")

# Put the quantcortex repo root on the path (notebooks live in research/).
for p in ("..", "."):
    ap = os.path.abspath(p)
    if ap not in sys.path:
        sys.path.insert(0, ap)

RNG = np.random.default_rng(42)

def load_prices(symbols, start="2018-01-01", periods=1800):
    """Real prices via yfinance if available, else deterministic synthetic GBM."""
    try:
        from quantcortex.data.providers.yfinance_provider import YFinanceProvider
        px = YFinanceProvider().get_prices(list(symbols), start=start)
        if px is not None and not px.empty and px.shape[0] > 120:
            return px.dropna(how="all").ffill().dropna()
        raise RuntimeError("empty frame")
    except Exception as exc:
        print(f"[offline] yfinance unavailable ({type(exc).__name__}); using synthetic data.")
    dates = pd.bdate_range(start, periods=periods)
    mu = RNG.normal(0.0003, 0.00015, len(symbols))
    vol = RNG.uniform(0.008, 0.018, len(symbols))
    shocks = RNG.normal(mu, vol, size=(periods, len(symbols)))
    return pd.DataFrame(100 * np.exp(np.cumsum(shocks, axis=0)),
                        index=dates, columns=list(symbols))

def synth_ohlcv(close):
    """Build an OHLCV frame around a close series (synthetic intrabar range)."""
    close = close.dropna()
    hi = close * (1 + np.abs(RNG.normal(0, 0.004, len(close))))
    lo = close * (1 - np.abs(RNG.normal(0, 0.004, len(close))))
    op = close.shift(1).fillna(close.iloc[0])
    vol = RNG.integers(1_000_000, 6_000_000, len(close)).astype(float)
    return pd.DataFrame({"open": op, "high": hi, "low": lo, "close": close, "volume": vol})

print("quantcortex research environment ready.")


quantcortex research environment ready.


In [2]:
symbols = ["QQQ","VGT","GLD","TLT","SPY","VIG"]
prices = load_prices(symbols)
from quantcortex.strategies.multi_asset_rotation import MultiAssetRotation

weekly = prices.index[prices.index.weekday == 0]
W = MultiAssetRotation().generate_weights(prices, weekly)
# The rotation strategy's regime gate flattens the book (all-zero) in bearish
# weeks; pick the most recent INVESTED rebalance so the execution walkthrough
# below actually has orders to route.
invested = W[W.abs().sum(axis=1) > 1e-9]
if invested.empty:
    raise RuntimeError("no invested rebalance in the sample to demo execution")
target = invested.iloc[-1]
target = target[target.abs() > 1e-9]
print(f"latest invested target weights (as of {target.name.date()}):")
print(target.round(3).to_string())

latest invested target weights (as of 2026-06-01):
GLD    0.489
TLT    0.011


## Pre-trade risk gate
The last line of defence before any order is routed.

In [3]:
from quantcortex.execution.pre_trade_risk import PreTradeRiskCheck
from quantcortex.portfolio.base import PortfolioMode

w_vec = target.reindex(symbols).fillna(0.0).to_numpy()
checker = PreTradeRiskCheck(max_position_weight=0.6, max_gross=1.0)
ok, violations = checker.check_weights(w_vec, mode=PortfolioMode.LONG_ONLY)
print("pre-trade weight check ok:", ok, "| violations:", violations)

pre-trade weight check ok: True | violations: []


## Translate target weights into orders

In [4]:
from quantcortex.execution.position_manager import PositionManager

capital = 100_000.0
last_px = prices.iloc[-1]
pm = PositionManager()
orders = pm.target_weights_to_orders(target, last_px, capital=capital, current_positions={})
for o in orders:
    print("  %4s %8.1f  %s" % (o["side"].value, o["quantity"], o["symbol"]))

   BUY    123.0  GLD
   BUY     12.0  TLT


## Order lifecycle: NEW -> SUBMITTED -> FILLED

In [5]:
from quantcortex.execution.order_manager import OrderManager, OrderSide

om = OrderManager()
for i, o in enumerate(orders):
    oid = f"ord-{i:03d}"
    om.create_order(oid, o["symbol"], OrderSide(o["side"]), float(o["quantity"]))
    om.submit(oid)
    om.fill(oid, fill_price=float(last_px[o["symbol"]]))
states = {o.order_id: o.status.value for o in om.orders}
print("final order states:", states)
assert states and all(s == "FILLED" for s in states.values())
print(f"all {len(states)} orders FILLED.")

final order states: {'ord-000': 'FILLED', 'ord-001': 'FILLED'}
all 2 orders FILLED.


## Persist state across restarts (Redis, file fallback offline)

In [6]:
from quantcortex.execution.state_persistence import StatePersistence

sp = StatePersistence(url=None)            # offline -> JSON file backend
positions = {o["symbol"]: float(o["quantity"]) for o in orders}
sp.save_positions(positions)
restored = sp.load_positions()
print("persistence backend:", sp.backend)
print("restored positions:", restored)

persistence backend: file
restored positions: {'GLD': Position(symbol='GLD', quantity=123.0, avg_price=0.0, market_price=0.0), 'TLT': Position(symbol='TLT', quantity=12.0, avg_price=0.0, market_price=0.0)}


## Alpaca paper connection (auto-skipped without credentials)

In [7]:
import os

from quantcortex.execution.brokers.alpaca_broker import AlpacaBroker

if os.getenv("ALPACA_API_KEY") and os.getenv("ALPACA_SECRET_KEY"):
    try:
        broker = AlpacaBroker(paper=True)
        broker.connect()
        acct = broker.get_account()
        print(f"Connected to Alpaca paper. Equity=${acct.equity:,.2f}")
    except Exception as e:
        print("Alpaca connection failed:", e)
else:
    print("No ALPACA_API_KEY set -> running offline.")
    print("Set credentials in .env to route this rebalance to Alpaca paper.")
print("\none rebalance cycle complete (paper/offline).")

No ALPACA_API_KEY set -> running offline.
Set credentials in .env to route this rebalance to Alpaca paper.

one rebalance cycle complete (paper/offline).
